# Home task: pandas 

## Question 1

- Load the energy data from the file [Energy Indicators.xls](http://unstats.un.org/unsd/environment/excel_file_tables/2013/Energy%20Indicators.xls).
It is a list of indicators of energy supply and renewable electricity production from the United Nations for the year 2013.


- It should be put into a DataFrame with the variable name of "energy"


- Make sure to exclude the footer and header information from the datafile.


- The first two columns are unneccessary, so you should get rid of them, and you should change the column labels so that the columns are:<br>
`['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable']`


- Convert `Energy Supply` to gigajoules (there are 1,000,000 gigajoules in a petajoule).


- For all countries which have missing data (e.g. data with `...`) make sure this is reflected as `np.NaN` values.


- Rename the following list of countries (for use in later questions):
    - `Republic of Korea`: `South Korea`,
    - `United States of America`: `United States`,
    - `United Kingdom of Great Britain and Northern Ireland`: `United Kingdom`,
    - `China, Hong Kong Special Administrative Region`: `Hong Kong`


- There are also several countries with numbers and/or parenthesis in their name. Be sure to remove these, e.g.:
    - `Bolivia (Plurinational State of)` should be `Bolivia`,
    - `Switzerland17` should be `Switzerland`.


- Next, load the GDP data from the file ["world_bank.csv"](http://data.worldbank.org/indicator/NY.GDP.MKTP.CD). 
It is a csv containing countries' GDP from 1960 to 2015 from World Bank. Call this DataFrame "GDP"


- Make sure to skip the header, and rename the following list of countries:
    - `Korea, Rep.`: `South Korea`,
    - `Iran, Islamic Rep.`: `Iran`,
    - `Hong Kong SAR, China`: `Hong Kong`


- Finally, load the "Sciamgo Journal and Country Rank data for [Energy Engineering and Power Technology"](http://www.scimagojr.com/countryrank.php?category=2102). It ranks countries based on their journal contributions in the aforementioned area. Call this DataFrame "ScimEn"


- Join the three datasets: Energy, GDP, and ScimEn into a new dataset (using the intersection of country names). Use only the 10 years (2006-2015) of GDP data and only the top 15 countries by Scimagojr 'Rank' (Rank 1 through 15).


- The index of this DataFrame should be the name of the country, and the columns should be<br>
`['Rank', 'Documents', 'Citable documents', 'Citations', 'Self-citations', 'Citations per document', 'H index', 'Energy Supply', 'Energy Supply per Capita', '% Renewable', '2006', '2007', '2008', '2009', '2010', '2011', 2012', '2013', '2014', '2015']`

Function "answer_one" should return the resulted DataFrame (20 columns and 15 entries)

## Answer the following questions in the context of only the top 15 countries by Scimagojr Rank (aka the DataFrame returned by `answer_one()`)

In [2]:
import pandas as pd
import numpy as np

In [5]:
def answer_one():
    
    # load energy data, skip the header junk and footer notes
    # '...' in the file means missing value, so treat it as NaN
    energy = (
        pd.read_excel("Energy Indicators.xls", usecols=[2,3,4,5],
                  skiprows=17, skipfooter=38, na_values="...")
        .set_axis(['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable'], axis=1)
        .assign(
            # original values are in petajoules, convert to gigajoules
            energy_supply=lambda df: df['Energy Supply'] * 1_000_000,
            # strip footnote numbers and parenthetical notes from country names
            Country=lambda df: df['Country']
                .str.replace(r"\s*\(.*\)", "", regex=True)
                .str.replace(r"\d+", "", regex=True)
                .str.strip()
            )
        )

    # some countries have different names across datasets, fix that
    country_rename = {"Republic of Korea": "South Korea",
        "United States of America": "United States",
        "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
        "China, Hong Kong Special Administrative Region": "Hong Kong"
    }
    energy['Country'] = energy['Country'].replace(country_rename)
    
    # load GDP data, first 4 rows are metadata so skip them
    GDP = pd.read_csv("world_bank.csv", skiprows=4)

    # same naming issue as above
    gdp_rename = {"Korea, Rep.": "South Korea",
        "Iran, Islamic Rep.": "Iran",
        "Hong Kong SAR, China": "Hong Kong"
    }

    GDP['Country Name'] = GDP['Country Name'].replace(gdp_rename)

    # only keep the years we actually need
    columns_to_keep = ['Country Name', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015']
    GDP = GDP[columns_to_keep]
    GDP = GDP.rename(columns={'Country Name': 'Country'})

    # load science rankings, we only care about the top 15
    ScimEn = pd.read_excel("scimagojr country rank 1996-2024 (1).xlsx")
    ScimEn_top15 = ScimEn.head(15)

    # merge all three sources on country name, then tidy up the column order
    df= (ScimEn_top15.merge(energy, on='Country', how='inner')
          .merge(GDP, on='Country', how='inner', suffixes=('', '_GDP'))
          .set_index('Country')
        [['Rank', 'Documents', 'Citable documents', 'Citations', 'Self-citations', 'Citations per document', 'H index',
          'Energy Supply', 'Energy Supply per Capita', '% Renewable',
          '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015']])
    
    return df

answer_one()


,Rank,Documents,Citable documents,Citations,Self-citations,Citations per document,H index,Energy Supply,Energy Supply per Capita,% Renewable,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
Country,,,,,,,,,,,,,,,,,,,,
China,1,472465,470142,6591474,4594673,13.95,380,127191.0,93.0,19.754910,2.752119e+12,3.550328e+12,4.594337e+12,5.101691e+12,6.087192e+12,7.551546e+12,8.532185e+12,9.570471e+12,1.047562e+13,1.106157e+13
United States,2,222772,217929,4131736,1119917,18.55,491,90838.0,286.0,11.570980,1.381558e+13,1.447423e+13,1.476986e+13,1.447807e+13,1.504897e+13,1.559973e+13,1.625397e+13,1.688068e+13,1.760814e+13,1.829502e+13
India,3,96975,94714,1243636,479638,12.82,258,33195.0,26.0,14.969080,9.402599e+11,1.216736e+12,1.198895e+12,1.341888e+12,1.675616e+12,1.823052e+12,1.827638e+12,1.856722e+12,2.039126e+12,2.103588e+12
United Kingdom,4,61946,60287,1326766,207337,21.42,313,7920.0,124.0,10.600470,2.708442e+12,3.090510e+12,2.929412e+12,2.412840e+12,2.485483e+12,2.663806e+12,2.707090e+12,2.784854e+12,3.064708e+12,2.927911e+12
Japan,5,61939,61307,813472,166907,13.13,246,18984.0,149.0,10.232820,4.601663e+12,4.579751e+12,5.106679e+12,5.289493e+12,5.759072e+12,6.233147e+12,6.272363e+12,5.212328e+12,4.896994e+12,4.444931e+12
Germany,6,55466,54296,932195,185376,16.81,274,13261.0,165.0,17.901530,3.046309e+12,3.484057e+12,3.808786e+12,3.479801e+12,3.468154e+12,3.824829e+12,3.597897e+12,3.808086e+12,3.965801e+12,3.423568e+12
Russian Federation,7,49938,49562,268391,112125,5.37,123,30709.0,214.0,17.288680,9.899321e+11,1.299703e+12,1.660848e+12,1.222646e+12,1.524917e+12,2.045923e+12,2.208294e+12,2.292470e+12,2.059242e+12,1.363482e+12
Canada,8,44877,44004,1082361,161133,24.12,305,10431.0,296.0,61.945430,1.319265e+12,1.468820e+12,1.552990e+12,1.374625e+12,1.617343e+12,1.793327e+12,1.828366e+12,1.846597e+12,1.805750e+12,1.556509e+12
Italy,9,42635,40695,767280,171442,18.00,227,6530.0,109.0,33.667230,1.958564e+12,2.222524e+12,2.417508e+12,2.209484e+12,2.144936e+12,2.306974e+12,2.097929e+12,2.153226e+12,2.173256e+12,1.845428e+12


### Question 2
What is the average GDP over the last 10 years for each country? (exclude missing values from this calculation.)

*This function should return a Series named `avgGDP` with 15 countries and their average GDP sorted in descending order.*

In [6]:
def answer_two():
    top15 = answer_one()

    # average GDP across all 10 years, then sort so highest is first
    top15['avgGDP'] = top15.loc[:, '2006':'2015'].mean(axis=1)
    top15_sorted = top15.sort_values(by='avgGDP', ascending=False)
    return top15_sorted['avgGDP']

answer_two()

Country
United States         1.572243e+13
China                 6.927707e+12
Japan                 5.239642e+12
Germany               3.590729e+12
United Kingdom        2.777505e+12
France                2.692000e+12
Italy                 2.152983e+12
Brazil                1.988889e+12
Russian Federation    1.666746e+12
Canada                1.616359e+12
India                 1.602352e+12
Spain                 1.406644e+12
South Korea           1.221328e+12
Australia             1.207997e+12
Iran                  4.567516e+11
Name: avgGDP, dtype: float64

### Question 3
By how much had the GDP changed over the 10 year span for the country with the 6th largest average GDP?

*This function should return a single number.*

In [7]:
def answer_three():
    Top15 = answer_one()
    avgGDP = answer_two()

    # grab the 6th country from the sorted list (index 5 since we start at 0)
    country_6 = avgGDP.index[5]

    # how much did their GDP grow between 2006 and 2015?
    gdp_change = Top15.loc[country_6, '2015'] - Top15.loc[country_6, '2006']
    return gdp_change

answer_three()

np.float64(124621907951.68018)

### Question 4

Create a new column that is the ratio of Self-Citations to Total Citations. 
What is the maximum value for this new column, and what country has the highest ratio?

*This function should return a tuple with the name of the country and the ratio.*

In [8]:
def answer_four():
    Top15 = answer_one()

    # what share of a country's citations are self-citations?
    Top15['Citations Ratio'] = Top15['Self-citations'] / Top15['Citations']

    # find who has the highest ratio
    max_ratio_country = Top15['Citations Ratio'].idxmax()
    max_ratio = Top15['Citations Ratio'].max()

    return (max_ratio_country, max_ratio)

answer_four()


('China', np.float64(0.6970630544852335))

### Question 5

Create a column that estimates the population using Energy Supply and Energy Supply per capita. 
What is the third most populous country according to this estimate?

*This function should return a single string value.*

In [9]:
def answer_five():
    Top15 = answer_one()

    # estimate population from energy data since we don't have it directly
    Top15['Population'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']

    # sort biggest to smallest, then pick the 3rd one
    mean_population = Top15['Population'].sort_values(ascending=False)

    return mean_population.index[2]

answer_five()

'United States'

### Question 6
Create a column that estimates the number of citable documents per person. 
What is the correlation between the number of citable documents per capita and the energy supply per capita? Use the `.corr()` method, (Pearson's correlation).

*This function should return a single number.*


In [10]:
def answer_six ():
    Top15 = answer_one()

    # need population first so we can calculate per-capita docs
    Top15['Population'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']
    Top15['Citable docs per Capita'] = Top15['Citable documents'] / Top15['Population']

    # do more productive countries use more energy? let's check the correlation
    return Top15['Citable docs per Capita'].corr(Top15['Energy Supply per Capita'])

answer_six()

np.float64(0.6905473831164102)

### Question 7
Use the following dictionary to group the Countries by Continent, then create a dateframe that displays the sample size (the number of countries in each continent bin), and the sum, mean, and std deviation for the estimated population of each country.

```python
ContinentDict  = {'China':'Asia', 
                  'United States':'North America', 
                  'Japan':'Asia', 
                  'United Kingdom':'Europe', 
                  'Russian Federation':'Europe', 
                  'Canada':'North America', 
                  'Germany':'Europe', 
                  'India':'Asia',
                  'France':'Europe', 
                  'South Korea':'Asia', 
                  'Italy':'Europe', 
                  'Spain':'Europe', 
                  'Iran':'Asia',
                  'Australia':'Australia', 
                  'Brazil':'South America'}
```

*This function should return a DataFrame with index named Continent `['Asia', 'Australia', 'Europe', 'North America', 'South America']` and columns `['size', 'sum', 'mean', 'std']`*

In [11]:
def answer_seven():
    Top15 = answer_one()

    # manually map each country to its continent
    ContinentDict  = {'China':'Asia', 
                  'United States':'North America', 
                  'Japan':'Asia', 
                  'United Kingdom':'Europe', 
                  'Russian Federation':'Europe', 
                  'Canada':'North America', 
                  'Germany':'Europe', 
                  'India':'Asia',
                  'France':'Europe', 
                  'South Korea':'Asia', 
                  'Italy':'Europe', 
                  'Spain':'Europe', 
                  'Iran':'Asia',
                  'Australia':'Australia', 
                  'Brazil':'South America'}
    
    # same population estimate as before
    Top15['Population'] = (Top15['Energy Supply'] / Top15['Energy Supply per Capita'])
    
    # assign continent to each row using the index (country name)
    Top15['Continent'] = Top15.index.to_series().map(ContinentDict)
    
    # group by continent and get basic population stats
    result = Top15.groupby('Continent')['Population'].agg(['size', 'sum', 'mean', 'std'])
    
    return result

answer_seven()
    
    

,size,sum,mean,std
Continent,,,,
Asia,5,2898.666387,579.733277,679.097888
Australia,1,23.316017,23.316017,NaN
Europe,6,457.929667,76.321611,34.647667
North America,2,352.855249,176.427625,199.669645
South America,1,205.915254,205.915254,NaN
